# M4.4: hybrid dense + sparse retrieval with RRF

This notebook compares frozen E5 dense, BM25 sparse, and rank-only Reciprocal Rank Fusion. It uses top-20 candidates from each retriever and primary `k=60`; it does not rerank or combine raw scores. If a CUDA failure occurred in this runtime, use **Runtime → Disconnect and delete runtime**, reopen the notebook, and Run All in a fresh session.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/ozgemelteminan/prompt-generator-rag'  # Replace this URL.
REPOSITORY_REF = 'main'  # Branch, tag, or commit to benchmark.
repository = Path('prompt-generator-rag')
if not repository.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL], check=True)
else:
    subprocess.run(['git', '-C', str(repository), 'fetch', '--all', '--tags', '--prune'], check=True)
subprocess.run(['git', '-C', str(repository), 'checkout', REPOSITORY_REF], check=True)
branch = subprocess.run(['git', '-C', str(repository), 'branch', '--show-current'], check=True, capture_output=True, text=True).stdout.strip()
if branch:
    subprocess.run(['git', '-C', str(repository), 'pull', '--ff-only', 'origin', branch], check=True)
os.chdir(repository)
subprocess.run(['pip', 'install', '-q', '--upgrade', 'transformers==4.57.6', 'sentence-transformers==5.6.0'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'packages/prompt-engine'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'apps/api', '--no-deps'], check=True)

import torch
import transformers
import sentence_transformers
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('GPU:', GPU_NAME)
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('sentence-transformers:', sentence_transformers.__version__)
assert transformers.__version__ == '4.57.6'
RUNTIME_METADATA = {'torchVersion': torch.__version__, 'transformersVersion': transformers.__version__, 'sentenceTransformersVersion': sentence_transformers.__version__, 'cudaDevice': GPU_NAME}

repository_root = Path.cwd().resolve()
api_root = repository_root / 'apps' / 'api'
for import_root in (repository_root, api_root):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))
stale_modules = [name for name in sys.modules if name == 'app' or name.startswith('app.') or name == 'evals' or name.startswith('evals.')]
if stale_modules:
    raise RuntimeError('Stale modules are loaded. Restart the runtime, then Run All.')

GPU: Tesla T4
torch: 2.11.0+cu128
transformers: 4.57.6
sentence-transformers: 5.6.0


In [2]:
import pandas as pd
from evals.src.dataset import load_dataset
from evals.src.embedding_eval import SentenceTransformerEmbeddingAdapter, embedding_model_registry, frozen_production_chunks
from evals.src.hybrid_eval import CANDIDATE_DEPTH, RRF_K, run_hybrid_benchmark, save_hybrid_results

ROOT = Path.cwd()
dataset = load_dataset(ROOT / 'evals/datasets/retrieval_eval_v1.json')
chunks = frozen_production_chunks(dataset)  # Generated once with 350/500/40 and reused everywhere.
e5_spec = embedding_model_registry()['multilingual_e5_large_instruct']
assert e5_spec.model_id == 'intfloat/multilingual-e5-large-instruct'
adapter = SentenceTransformerEmbeddingAdapter(e5_spec)
try:
    evaluation = run_hybrid_benchmark(dataset, chunks=chunks, adapter=adapter, rrf_k=RRF_K, candidate_depth=CANDIDATE_DEPTH)
finally:
    adapter.release()
save_hybrid_results(evaluation, dataset_version=dataset.version, output_dir=ROOT / 'evals/results/hybrid', runtime_metadata=RUNTIME_METADATA)
results = evaluation.results

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

In [3]:
rows = []
for result in results:
    row = {'Retriever': result.retriever, **result.metrics}
    row['TR MRR'] = result.by_language.get('tr', {}).get('mrr', 0.0)
    row['EN MRR'] = result.by_language.get('en', {}).get('mrr', 0.0)
    rows.append(row)
comparison = pd.DataFrame(rows)
display(comparison)

hybrid = next(result for result in results if result.retriever_key == 'hybrid_rrf')
deltas = []
for baseline_key in ['dense_e5', 'sparse_bm25']:
    baseline = next(result for result in results if result.retriever_key == baseline_key)
    deltas.append({'Comparison': f'Hybrid - {baseline.retriever}', **{metric: hybrid.metrics[metric] - baseline.metrics[metric] for metric in hybrid.metrics}})
display(pd.DataFrame(deltas))

language_rows = []
for language in ['tr', 'en']:
    language_rows.append({'Language': language, **{result.retriever: result.by_language.get(language, {}).get('mrr', 0.0) for result in results}})
display(pd.DataFrame(language_rows))

category_rows = []
for category in ['factual', 'hard_paraphrase', 'terminology_mismatch', 'morphology_heavy', 'near_negative', 'same_topic_competitor', 'multi_section', 'cross_paragraph']:
    category_rows.append({'Category': category, **{result.retriever: result.by_category.get(category, {}).get('mrr', 0.0) for result in results}})
display(pd.DataFrame(category_rows))

for signal, query_ids in evaluation.diagnostics.items():
    print(f'{signal}: {len(query_ids)} queries', query_ids)

,Retriever,recall_at_5,recall_at_10,hit_rate_at_5,mrr,ndcg_at_10,required_block_coverage_at_5,required_block_coverage_at_10,TR MRR,EN MRR
0,Dense — intfloat/multilingual-e5-large-instruct,0.958333,1.000000,0.964286,0.870040,0.851617,0.958333,1.000000,0.940476,0.799603
1,Sparse — BM25,0.910714,0.958333,0.916667,0.844577,0.825343,0.910714,0.958333,0.938492,0.750661
2,Hybrid — Dense + BM25 RRF,0.958333,1.000000,0.964286,0.877395,0.856806,0.958333,1.000000,0.961905,0.792885


,Comparison,recall_at_5,recall_at_10,hit_rate_at_5,mrr,ndcg_at_10,required_block_coverage_at_5,required_block_coverage_at_10
0,Hybrid - Dense — intfloat/multilingual-e5-larg...,0.000000,0.000000,0.000000,0.007355,0.005189,0.000000,0.000000
1,Hybrid - Sparse — BM25,0.047619,0.041667,0.047619,0.032818,0.031463,0.047619,0.041667


,Language,Dense — intfloat/multilingual-e5-large-instruct,Sparse — BM25,Hybrid — Dense + BM25 RRF
0,tr,0.940476,0.938492,0.961905
1,en,0.799603,0.750661,0.792885


,Category,Dense — intfloat/multilingual-e5-large-instruct,Sparse — BM25,Hybrid — Dense + BM25 RRF
0,factual,0.884259,0.900463,0.910714
1,hard_paraphrase,1.000000,0.925926,0.958333
2,terminology_mismatch,0.513889,0.479167,0.511111
3,morphology_heavy,1.000000,1.000000,1.000000
4,near_negative,0.805556,0.722222,0.833333
5,same_topic_competitor,0.805556,0.847222,0.777778
6,multi_section,0.916667,1.000000,1.000000
7,cross_paragraph,0.652778,0.472222,0.595833


hybrid_improves_dense: 8 queries ['m42-security-02', 'm42-security-04', 'm42-security-14', 'm42-retrieval-04', 'm42-retrieval-09', 'm42-retrieval-10', 'm42-code-04', 'm42-code-08']
hybrid_hurts_dense: 6 queries ['m42-security-07', 'm42-code-03', 'm42-code-07', 'm42-planning-04', 'm42-planning-09', 'm42-planning-14']
hybrid_recovers_dense_miss: 0 queries []
bm25_only_signal: 0 queries []
dense_only_signal: 3 queries ['m42-code-07', 'm42-planning-09', 'm42-planning-14']


In [4]:
from google.colab import files

files.download(
    "/content/prompt-generator-rag/evals/results/hybrid/hybrid_rrf_results_v1.json"
)
files.download(
    "/content/prompt-generator-rag/evals/results/hybrid/hybrid_rrf_results_v1.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>